# Imbalanced_data_with_Pipeline_SMOTE


https://towardsdatascience.com/the-right-way-of-using-smote-with-cross-validation-92a8d09d00c7

- Use SMOTE to avoid inaccurate evaluation metrics while using cross-validation techniques

- Using SMOTE seperately is not a good option, we better to add all in pipeline, 

- SMOTE dont have fit_transform, so we cant use SKLEARN Pipeline here, so here we used imbpipeline

- Creating a pipeline AND Applied Grid Serach CV with Strarified K-Fold Crossvalidation and Minmax scalar

- We’ve used SMOTE as a part of a pipeline. This pipeline is not a ‘Scikit-Learn’ pipeline, but ‘imblearn’ pipeline. 

- Since, SMOTE doesn’t have a ‘fit_transform’ method, we cannot use it with ‘Scikit-Learn’ pipeline.

- Instead of using SMOTE seperately, if we use with Pipeline a major difference between the cross-validation scores of the trainset and set test. 

- In this example this difference is very low, might have happened just by chance and also may be because the dataset is not highly imbalanced.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as imbpipeline
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification, load_breast_cancer


X = load_breast_cancer()['data'].copy()
y = load_breast_cancer()['target'].copy()

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    stratify=y,
                                                    random_state=11)

#Apply smote seperately is not good option
#smote = SMOTE(random_state = 11)
#X_train, y_train = smote.fit_resample(X_train, y_train)

# Creating a pipeline
#pipeline = Pipeline(steps = [['scaler', MinMaxScaler()],
#                             ['classifier', LogisticRegression(random_state=11,
                                                             #  max_iter=1000)]])

pipeline = imbpipeline(steps = [['smote', SMOTE(random_state=11)],
                                ['scaler', MinMaxScaler()],
                                ['classifier', LogisticRegression(random_state=11,
                                                                  max_iter=1000)]])


# KFold - Stratified KFold, For imbalanced data, good to use stratified instead of normal one
stratified_kfold = StratifiedKFold(n_splits=3,  # 3 split
                                       shuffle=True,
                                       random_state=11)
    
param_grid = {'classifier__C':[0.001, 0.01, 0.1, 1, 10, 100, 1000]}

#Here 1st smote, MinMaxScaler applied on X data  --> Then applied Log Reg, --> Then stratified_kfold --> Then scoring ='ROC_AUC' and used param_grid

grid_search = GridSearchCV(estimator=pipeline,
                           param_grid=param_grid,
                           scoring='roc_auc',
                           cv=stratified_kfold,  # Normally hardcode to cv=10 etc
                           n_jobs=-1)


grid_search.fit(X_train, y_train)
cv_score = grid_search.best_score_
test_score = grid_search.score(X_test, y_test)
print(f'Cross-validation score: {cv_score}\nTest score: {test_score}')

Cross-validation score: 0.9972013366750209
Test score: 0.9798280423280423
